# Project 2: Build a Quiz Game

**Level**: Intermediate  
**Skills covered**: OOP, file I/O, JSON, `random`, `time`, list comprehensions, exception handling

---

## Project Goal

You will build a multiple-choice quiz game from scratch. The game will:
- Load questions from a JSON file (or fall back to built-in data)
- Shuffle questions and answer options randomly
- Track scores and enforce a per-question time limit
- Save high scores to a file
- Support category filtering so players can focus on a topic

---

## Steps Overview

1. Question data structure
2. `Question` class
3. `Quiz` class
4. Shuffling questions and options
5. Saving and loading questions from JSON
6. High scores
7. Category filtering
8. Timer per question

---

## Step 1: Question Data Structure

Before writing any classes, decide what a question looks like as plain data.

In [ ]:
# A question is a dictionary with four keys.
# This is the 'schema' we will enforce throughout the project.

sample_question = {
    "question": "What is the output of: print(type([])).__name__?",
    "options": ["dict", "list", "tuple", "set"],
    "answer": "list",
    "category": "Python Basics",
}

print(sample_question["question"])
print("Options:", sample_question["options"])
print("Correct answer:", sample_question["answer"])
print("Category:", sample_question["category"])

In [ ]:
# Built-in question bank used as fallback when no file is present

QUESTION_BANK = [
    {
        "question": "Which keyword is used to define a function in Python?",
        "options": ["func", "def", "lambda", "define"],
        "answer": "def",
        "category": "Python Basics",
    },
    {
        "question": "What does len([1, 2, 3]) return?",
        "options": ["2", "3", "4", "1"],
        "answer": "3",
        "category": "Python Basics",
    },
    {
        "question": "Which data structure uses key-value pairs?",
        "options": ["list", "tuple", "set", "dict"],
        "answer": "dict",
        "category": "Data Structures",
    },
    {
        "question": "What is the result of 2 ** 10?",
        "options": ["20", "100", "1024", "512"],
        "answer": "1024",
        "category": "Python Basics",
    },
    {
        "question": "Which module provides random number generation?",
        "options": ["math", "os", "random", "sys"],
        "answer": "random",
        "category": "Standard Library",
    },
    {
        "question": "What does a list comprehension return?",
        "options": ["a generator", "a list", "a tuple", "a dict"],
        "answer": "a list",
        "category": "Python Basics",
    },
    {
        "question": "Which keyword skips the rest of a loop iteration?",
        "options": ["break", "pass", "continue", "return"],
        "answer": "continue",
        "category": "Control Flow",
    },
    {
        "question": "What does dict.get('key', 'default') return if key is missing?",
        "options": ["None", "KeyError", "'default'", "False"],
        "answer": "'default'",
        "category": "Data Structures",
    },
    {
        "question": "Which built-in converts an integer to a string?",
        "options": ["int()", "str()", "repr()", "chr()"],
        "answer": "str()",
        "category": "Python Basics",
    },
    {
        "question": "What is the time complexity of list.append()?",
        "options": ["O(n)", "O(log n)", "O(1)", "O(n^2)"],
        "answer": "O(1)",
        "category": "Data Structures",
    },
]

print(f"Question bank loaded: {len(QUESTION_BANK)} questions")
categories = {q['category'] for q in QUESTION_BANK}
print(f"Categories: {categories}")

---

## Step 2: The `Question` Class

Wrap the dictionary in a class for type safety, display helpers, and answer checking.

In [ ]:
class Question:
    """Represents a single multiple-choice question."""

    def __init__(self, question, options, answer, category="General"):
        if answer not in options:
            raise ValueError(f"Answer '{answer}' is not in options {options}")
        self.question = question
        self.options = list(options)   # copy so we can shuffle safely
        self.answer = answer
        self.category = category

    @classmethod
    def from_dict(cls, data):
        """Create a Question from a dictionary (matches QUESTION_BANK schema)."""
        return cls(
            question=data["question"],
            options=data["options"],
            answer=data["answer"],
            category=data.get("category", "General"),
        )

    def to_dict(self):
        """Serialize back to a dictionary (for JSON saving)."""
        return {
            "question": self.question,
            "options": self.options,
            "answer": self.answer,
            "category": self.category,
        }

    def is_correct(self, user_answer):
        """Return True if user_answer matches the correct answer (case-insensitive)."""
        return user_answer.strip().lower() == self.answer.strip().lower()

    def display(self, shuffled_options=None):
        """Print the question with labelled options (A/B/C/D)."""
        options = shuffled_options or self.options
        print(f"\n{self.question}")
        for label, opt in zip("ABCD", options):
            print(f"  {label}) {opt}")

    def __repr__(self):
        return f"Question(category={self.category!r}, q={self.question[:40]!r})"


# Test the class
q = Question.from_dict(QUESTION_BANK[0])
print(repr(q))
q.display()
print(f"\nCorrect answer: {q.answer}")
print(f"is_correct('def') -> {q.is_correct('def')}")
print(f"is_correct('func') -> {q.is_correct('func')}")

**Key concepts**: `@classmethod`, `from_dict` factory pattern, `__repr__`, defensive validation in `__init__`.

---

## Step 3: The `Quiz` Class

The `Quiz` class loads questions, manages the game state (score, current index), and drives the flow.

In [ ]:
class Quiz:
    """Manages a quiz session: loading, scoring, and running questions."""

    def __init__(self, questions):
        """
        questions: list of Question objects (or list of dicts — we convert).
        """
        self.questions = [
            q if isinstance(q, Question) else Question.from_dict(q)
            for q in questions
        ]
        self.score = 0
        self.answers_given = []   # list of (Question, user_answer, correct)

    def reset(self):
        """Reset score and answer history for a fresh run."""
        self.score = 0
        self.answers_given = []

    def percentage(self):
        """Return score as a percentage of questions answered."""
        total = len(self.answers_given)
        return (self.score / total * 100) if total else 0

    def grade(self):
        """Return a letter grade based on percentage."""
        pct = self.percentage()
        if pct >= 90: return "A"
        if pct >= 80: return "B"
        if pct >= 70: return "C"
        if pct >= 60: return "D"
        return "F"

    def summary(self):
        """Print a results summary."""
        total = len(self.answers_given)
        print("\n" + "=" * 40)
        print(f"  Quiz Complete!")
        print(f"  Score : {self.score} / {total}")
        print(f"  Pct   : {self.percentage():.1f}%")
        print(f"  Grade : {self.grade()}")
        print("=" * 40)

        # Show which answers were wrong
        wrong = [(q, ua) for q, ua, correct in self.answers_given if not correct]
        if wrong:
            print("\nReview — questions you missed:")
            for q, ua in wrong:
                print(f"  Q: {q.question}")
                print(f"     Your answer  : {ua}")
                print(f"     Correct answer: {q.answer}")


# Quick smoke test
quiz = Quiz(QUESTION_BANK)
print(f"Quiz created with {len(quiz.questions)} questions")
print(f"Categories in this quiz: {sorted({q.category for q in quiz.questions})}")

---

## Step 4: Shuffling Questions and Answer Options

Use the `random` module to shuffle both the question order and the option order each run.

In [ ]:
import random

def shuffle_questions(questions):
    """Return a shuffled copy of the question list."""
    shuffled = questions[:]   # shallow copy
    random.shuffle(shuffled)
    return shuffled


def shuffle_options(question):
    """
    Return (shuffled_options, correct_label) where correct_label is
    the letter (A/B/C/D) of the correct answer after shuffling.
    """
    options = question.options[:]
    random.shuffle(options)
    labels = "ABCD"
    correct_label = labels[options.index(question.answer)]
    return options, correct_label


# Demo: run the same question three times with different shuffles
q = Question.from_dict(QUESTION_BANK[2])
print(f"Question: {q.question}")
print(f"Correct answer: {q.answer}\n")

for run in range(3):
    opts, label = shuffle_options(q)
    print(f"  Shuffle {run+1}:")
    for l, o in zip("ABCD", opts):
        marker = " <-- correct" if o == q.answer else ""
        print(f"    {l}) {o}{marker}")
    print(f"  -> Correct label: {label}")
    print()

**Key concepts**: `list[:]` for shallow copy, `random.shuffle`, `list.index`.

---

## Step 5: Saving and Loading Questions from JSON

Questions should live in a file, not hardcoded in the source.

In [ ]:
import json
import os

QUESTIONS_FILE = "quiz_questions.json"

def save_questions(questions, filepath=QUESTIONS_FILE):
    """Save a list of Question objects (or dicts) to a JSON file."""
    data = [
        q.to_dict() if isinstance(q, Question) else q
        for q in questions
    ]
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(data)} questions to '{filepath}'")


def load_questions(filepath=QUESTIONS_FILE):
    """
    Load questions from a JSON file.
    Falls back to QUESTION_BANK if file doesn't exist.
    """
    if not os.path.exists(filepath):
        print(f"'{filepath}' not found — using built-in question bank.")
        return [Question.from_dict(d) for d in QUESTION_BANK]

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        questions = [Question.from_dict(d) for d in data]
        print(f"Loaded {len(questions)} questions from '{filepath}'")
        return questions
    except (json.JSONDecodeError, KeyError) as e:
        print(f"Error reading '{filepath}': {e} — using built-in bank.")
        return [Question.from_dict(d) for d in QUESTION_BANK]


# Save the built-in bank, then reload it
save_questions(QUESTION_BANK)
loaded = load_questions()
print(f"Round-trip check: {len(loaded)} questions loaded back successfully.")

**Key concepts**: `json.dump` / `json.load`, `os.path.exists`, graceful fallback, `encoding="utf-8"`.

---

## Step 6: High Scores

Persist the top scores across sessions using a JSON file.

In [ ]:
import json
import os
from datetime import datetime

SCORES_FILE = "quiz_highscores.json"
MAX_HIGH_SCORES = 10


def load_high_scores(filepath=SCORES_FILE):
    if not os.path.exists(filepath):
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def save_high_score(name, score, total, category="All", filepath=SCORES_FILE):
    """Append a new score and keep only the top MAX_HIGH_SCORES."""
    scores = load_high_scores(filepath)
    entry = {
        "name": name,
        "score": score,
        "total": total,
        "percentage": round(score / total * 100, 1) if total else 0,
        "category": category,
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    scores.append(entry)
    # Sort by percentage descending, then by date descending as tiebreaker
    scores.sort(key=lambda s: (-s["percentage"], s["date"]), reverse=False)
    scores = scores[:MAX_HIGH_SCORES]
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(scores, f, indent=2)
    return scores


def display_high_scores(filepath=SCORES_FILE):
    scores = load_high_scores(filepath)
    if not scores:
        print("No high scores yet.")
        return
    print("\n" + "=" * 52)
    print(f"  {'Rank':<5} {'Name':<15} {'Score':<10} {'Pct':>6}  {'Date'}")
    print("-" * 52)
    for i, s in enumerate(scores, 1):
        print(f"  {i:<5} {s['name']:<15} {s['score']}/{s['total']:<7} {s['percentage']:>5.1f}%  {s['date']}")
    print("=" * 52 + "\n")


# Simulate saving a few scores
save_high_score("Alice", 9, 10)
save_high_score("Bob",   7, 10)
save_high_score("Carol", 10, 10)
display_high_scores()

---

## Step 7: Category Filtering

Let the player choose to quiz on a specific category.

In [ ]:
def filter_by_category(questions, category):
    """
    Return only questions matching the given category.
    'All' returns every question.
    """
    if category.lower() == "all":
        return questions
    return [q for q in questions if q.category.lower() == category.lower()]


def list_categories(questions):
    """Return a sorted list of unique categories."""
    return sorted({q.category for q in questions})


# Demo
all_questions = load_questions()
cats = list_categories(all_questions)
print(f"Available categories: {cats}")

for cat in cats:
    filtered = filter_by_category(all_questions, cat)
    print(f"  {cat}: {len(filtered)} questions")

# Filter to one category
basics = filter_by_category(all_questions, "Python Basics")
print(f"\nPython Basics questions:")
for q in basics:
    print(f"  - {q.question}")

**Key concepts**: list comprehension with condition, set comprehension for unique values.

---

## Step 8: Timer Per Question

Enforce a time limit using the `time` module.

In [ ]:
import time

def ask_with_timer(question, time_limit=15):
    """
    Display a question, shuffle its options, and collect an answer.
    Returns (user_answer_str, elapsed_seconds, timed_out: bool).

    Note: True non-blocking timers require threading; this version uses
    simple elapsed-time checking and is compatible with Jupyter.
    In a terminal script you would use 'signal' or 'threading' for hard cutoffs.
    """
    shuffled_opts, correct_label = shuffle_options(question)
    question.display(shuffled_options=shuffled_opts)
    print(f"  [Time limit: {time_limit}s]")

    start = time.time()
    answer = input("Your answer (A/B/C/D): ").strip().upper()
    elapsed = time.time() - start

    timed_out = elapsed > time_limit
    if timed_out:
        print(f"  Too slow! ({elapsed:.1f}s > {time_limit}s limit)")
        return "", elapsed, True

    # Map label back to the actual option text
    labels = "ABCD"
    if answer in labels:
        chosen_text = shuffled_opts[labels.index(answer)]
    else:
        chosen_text = answer  # let is_correct handle the mismatch

    return chosen_text, elapsed, False


# Demonstrate the timer logic without blocking (simulate fast input)
print("Timer demo — check that elapsed time is measured correctly.")
start_demo = time.time()
time.sleep(0.1)  # simulate very fast 'user'
elapsed_demo = time.time() - start_demo
print(f"Simulated response in {elapsed_demo:.3f}s  (well within any reasonable limit)")

**Key concepts**: `time.time()`, elapsed timing, mapping letter labels to option text.

---

## Final: Full Interactive Quiz Runner

Combine everything into a single `run_quiz()` function.

In [ ]:
def run_quiz(
    questions=None,
    category="All",
    time_limit=15,
    player_name="Player",
    shuffle=True,
):
    """
    Run a full quiz session.

    Parameters
    ----------
    questions  : list of Question or None (loads from file/bank)
    category   : category filter string, or 'All'
    time_limit : seconds allowed per question
    player_name: displayed in high score
    shuffle    : whether to shuffle question and option order
    """
    # Load
    if questions is None:
        questions = load_questions()

    # Filter
    questions = filter_by_category(questions, category)
    if not questions:
        print(f"No questions found for category '{category}'.")
        return

    # Shuffle
    if shuffle:
        questions = shuffle_questions(questions)

    quiz = Quiz(questions)

    print(f"\nWelcome, {player_name}! Category: {category} | {len(questions)} questions")
    print("Type 'quit' at any time to stop.\n")

    for i, q in enumerate(quiz.questions, 1):
        print(f"Question {i} of {len(quiz.questions)}  [Category: {q.category}]")

        shuffled_opts, _ = shuffle_options(q)
        q.display(shuffled_options=shuffled_opts)
        print(f"  [Time limit: {time_limit}s]")

        start = time.time()
        try:
            raw = input("Your answer (A/B/C/D or 'quit'): ").strip().upper()
        except (KeyboardInterrupt, EOFError):
            print("\nQuiz interrupted.")
            break

        elapsed = time.time() - start

        if raw == "QUIT":
            print("Quitting early.")
            break

        if elapsed > time_limit:
            print(f"  Time's up! ({elapsed:.1f}s). Correct answer: {q.answer}")
            quiz.answers_given.append((q, "", False))
            continue

        # Map A/B/C/D back to option text
        labels = "ABCD"
        if raw in labels:
            user_answer = shuffled_opts[labels.index(raw)]
        else:
            user_answer = raw

        correct = q.is_correct(user_answer)
        quiz.answers_given.append((q, user_answer, correct))

        if correct:
            quiz.score += 1
            print(f"  Correct! ({elapsed:.1f}s)")
        else:
            print(f"  Wrong. Correct answer: {q.answer}  ({elapsed:.1f}s)")

    quiz.summary()
    save_high_score(player_name, quiz.score, len(quiz.answers_given), category)
    display_high_scores()


# Uncomment to play interactively:
# run_quiz(player_name="Alice", category="Python Basics", time_limit=20)

print("run_quiz() is ready. Uncomment the last line to play!")

---

## Summary

| Step | What you built | Skills |
|------|---------------|--------|
| 1 | Question data structure | dicts, list of dicts |
| 2 | `Question` class | OOP, `@classmethod`, `__repr__` |
| 3 | `Quiz` class | OOP, state management |
| 4 | Shuffling | `random.shuffle`, shallow copy |
| 5 | JSON persistence | `json.dump/load`, file I/O |
| 6 | High scores | sorting, `datetime` |
| 7 | Category filter | list comprehension, set comprehension |
| 8 | Timer | `time.time()`, elapsed timing |

---

## How to Extend This

- **Difficulty levels**: add a `difficulty` field (`easy/medium/hard`) to Question and filter by it
- **Multiplayer**: run `run_quiz()` in a loop for multiple players and compare scores at the end
- **Import from CSV**: write a `load_questions_csv()` that reads a spreadsheet of questions
- **Web version**: serve the quiz as a Flask app with an HTML front-end and session-based scoring
- **Explanations**: add an `explanation` field to Question and display it after each wrong answer
- **Spaced repetition**: track how many times each question was answered wrong and ask those more often